# 第 6 课：工作、会话与长期记忆

预计用时：75–90 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 区分三种记忆的生命周期
- 用 SQLite 保存偏好与文本记忆
- 用向量相似度召回相关内容

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 1. 先理解概念

工作记忆只服务当前执行；会话记忆借助 checkpoint 跨轮保存状态；长期记忆跨会话保存用户偏好与经验。长期记忆必须有用户边界、来源、置信度、删除机制和隐私策略。

### 本课路线

1. 创建两张 SQLite 表
2. 写入可更新的结构化偏好
3. 调用 Embedding 生成向量
4. 把 float32 向量保存为 BLOB
5. 计算余弦相似度并返回 Top-K


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 准备代码


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 核心实验


In [ ]:
MEMORY_DB = WORKSPACE / 'memory.sqlite3'

class LongTermMemory:
    def __init__(self, path: Path = MEMORY_DB) -> None:
        self.path = path
        with sqlite3.connect(self.path) as conn:
            conn.executescript('''
                CREATE TABLE IF NOT EXISTS preferences (
                    user_id TEXT, key TEXT, value TEXT, confidence REAL, source TEXT, updated_at TEXT,
                    PRIMARY KEY(user_id, key)
                );
                CREATE TABLE IF NOT EXISTS memories (
                    id INTEGER PRIMARY KEY, user_id TEXT, text TEXT, embedding BLOB, created_at TEXT
                );
            ''')

    def upsert_preference(self, user_id: str, key: str, value: str, source: str, confidence: float = 1.0) -> None:
        if not 0 <= confidence <= 1:
            raise ValueError('confidence 必须在 0 到 1 之间')
        with sqlite3.connect(self.path) as conn:
            conn.execute('''
                INSERT INTO preferences VALUES (?, ?, ?, ?, ?, ?)
                ON CONFLICT(user_id, key) DO UPDATE SET
                  value=excluded.value, confidence=excluded.confidence,
                  source=excluded.source, updated_at=excluded.updated_at
            ''', (user_id, key, value, confidence, source, datetime.now(timezone.utc).isoformat()))

    def preferences(self, user_id: str) -> dict[str, str]:
        with sqlite3.connect(self.path) as conn:
            rows = conn.execute('SELECT key, value FROM preferences WHERE user_id=?', (user_id,)).fetchall()
        return dict(rows)

    async def embed(self, texts: list[str]) -> list[list[float]]:
        require_api_key()
        response = await client.embeddings.create(model=BAILIAN_EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    async def remember(self, user_id: str, text: str) -> int:
        vector = np.asarray((await self.embed([text]))[0], dtype=np.float32)
        with sqlite3.connect(self.path) as conn:
            cursor = conn.execute(
                'INSERT INTO memories(user_id, text, embedding, created_at) VALUES (?, ?, ?, ?)',
                (user_id, text, vector.tobytes(), datetime.now(timezone.utc).isoformat()),
            )
            return int(cursor.lastrowid)

    async def recall(self, user_id: str, query: str, k: int = 5) -> list[dict[str, Any]]:
        query_vec = np.asarray((await self.embed([query]))[0], dtype=np.float32)
        with sqlite3.connect(self.path) as conn:
            rows = conn.execute('SELECT id, text, embedding FROM memories WHERE user_id=?', (user_id,)).fetchall()
        scored = []
        for memory_id, text, blob in rows:
            vector = np.frombuffer(blob, dtype=np.float32)
            score = float(np.dot(query_vec, vector) / (np.linalg.norm(query_vec) * np.linalg.norm(vector) + 1e-12))
            scored.append({'id': memory_id, 'text': text, 'score': score})
        return sorted(scored, key=lambda item: item['score'], reverse=True)[:k]

memory = LongTermMemory()
memory.upsert_preference('demo-user', 'travel_style', '喜欢安静、人少、步行友好的地方', '用户明确表达')
print(memory.preferences('demo-user'))
# await memory.remember('demo-user', '上次去苏州时更喜欢园林，不喜欢排队很久的网红店。')
# print(await memory.recall('demo-user', '周末去哪里比较合适？'))


## 3. 观察与验证

核心代码中的真实 API 调用默认被注释。先运行无需额度的断言或定义单元格；确认输出和预期一致后，再逐行取消示例注释。


## 4. 代码讲解

偏好使用 `(user_id, key)` 作为唯一键；自由文本记忆使用向量检索。示例用 SQLite 便于学习，不代表生产环境的最佳存储方案。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 为偏好增加删除方法
- 写入三条记忆并用不同问题检索
- 在 recall 中加入最低相似度阈值

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 能解释三层记忆的差别
- [ ] 不同 user_id 的数据不会混查
- [ ] 知道向量 BLOB 需要固定 dtype

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `07_Agentic_RAG.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。
